In [1]:
import json
from monty.json import MontyDecoder
import os
from pymatgen.core import Element

from utils_kga.statistical_analysis.analyze_ion_pairs import *

In [2]:
# Load edge-df
with open("data/dfs_of_magnetic_edge_information.json") as f:
# replace to analyze all structs, not only cryst. uniques:
#with open("data/dfs_of_magnetic_edge_information_include_crystallographic_multiples.json") as f:
    dict_all_stats = json.load(f)
all_stats = {key: pd.DataFrame.from_dict(df) for key, df in dict_all_stats.items()}

# For metadata filtering and computation of occurrences in magnetic primitive cells
with open("../../data_retrieval_and_preprocessing_MAGNDATA/data/df_grouped_and_chosen_commensurate_MAGNDATA.json") as f:
    df = json.load(f, cls=MontyDecoder)

plot_dir = "plots/TM_octahedra_ion_pair_distributions_at_corner_connections"
os.makedirs(plot_dir, exist_ok=True)

In [3]:
custom_sort_by_electron_configuration = [ 
    ("V", 3), ("Re", 5), ("Os", 6),
    ("Cr", 3), ("Mn", 4), ("Ru", 5), ("Os", 5),
    ("Mn", 3),
    ("Ru", 4),
    ("Mn", 2), ("Fe", 3), ("Co", 4),
    ("Ru", 3), ("Ir", 4),
    ("Mn", 1), ("Fe", 2), ("Co", 3),
    ("Co", 2), ("Ni", 3),
    ("Ni", 2),
    ("Cu", 2)
]

In [4]:
for ang_df in all_stats.values():
    ang_df["site_is_tm"] = ang_df["site_element"].apply(lambda el: Element(el).is_transition_metal)
    ang_df["site_to_is_tm"] = ang_df["site_to_element"].apply(lambda el: Element(el).is_transition_metal)
    ang_df["ligand_el_set"] = ang_df["ligand_elements"].apply(lambda ls: set(ls))



In [5]:
write_mode = "w"
ba_threshold = 150
normalize_bool = False
normalize_string = "absolute occurrences"

data_string = f"corner connections with TM octahedra at both nodes"
print(data_string)
occus = {"afm": {}, "fm": {}}
num_structures = {}  # track in how many structures ion pair is found with these conditions

missed = set()

for md_id, ang_df in all_stats.items():
    subdf = ang_df.loc[(ang_df["site_is_tm"])  & (ang_df["site_to_is_tm"])]
    subdf = subdf.loc[subdf[f"connectivity"]=="corner"]
    subdf = subdf.loc[(subdf["site_ce"]=="O:6")
            & (subdf["site_to_ce"]=="O:6")]

    if not subdf.empty:
        n_lattice_points = df.at[md_id, "n_lattice_points"]
        for mag_type, condition in zip(["afm", "fm", ],
                                        [(subdf["spin_angle"]>170), (subdf["spin_angle"]<10)]):
            type_df = subdf.loc[condition]
            if not type_df.empty:
                abs_ion_pairs = get_ion_pair_occus(df=type_df, n_lattice_points=n_lattice_points, normalize_bool=normalize_bool)

                for k, v in abs_ion_pairs.items():
                    # Assert no detected ions are missing
                    try:
                        assert (k[0], k[1]) in custom_sort_by_electron_configuration, str((k[0], k[1]))
                        assert (k[2], k[3]) in custom_sort_by_electron_configuration, str((k[2], k[3]))
                    except AssertionError as e:
                        missed.add(str(e))

                    if k in occus[mag_type]:
                        occus[mag_type][k] += v
                    else:
                        occus[mag_type][k] = v

                    if k in num_structures:
                        num_structures[k].add(md_id)
                    else:
                        num_structures[k] = {md_id}



print(occus)

print("missed", missed)

# Plot heatmap of FM/AFM ratio per ion pair (logarithmic, diverging color scale)
ratio_fig = plot_ion_pair_occurrence_ratio(occus=occus, ion_list=custom_sort_by_electron_configuration)
ratio_fig.update_layout(title=dict(
    text=f"Logarithmic plot of FM/AFM ratios, {data_string}, {normalize_string}",
    font=dict(size=10)))
# ratio_fig.show()
ratio_fig.write_html(os.path.join(plot_dir, f"log_plot_afm-fm-ratio_ion_pair_{normalize_string.replace(' ', '_')}_{data_string.replace(' ', '_').replace('°', 'deg')}.html"))

close_to_1_tracker, all_tracker = set(), set()
print("FM to AFM ratio  :")

for ip in occus["fm"]:
    ip_string = "".join(sorted([str(ip[0])+str(ip[1]), str(ip[2])+str(ip[3])]))
    if ip in occus["afm"]:
        print(ip, round(occus["fm"][ip] / occus["afm"][ip], 2), round(log10(occus["fm"][ip] / occus["afm"][ip]), 2), len(num_structures[ip]))
        if abs(log10(occus["fm"][ip] / occus["afm"][ip])) < 0.5:
            close_to_1_tracker.add(ip_string)
        all_tracker.add(ip_string)
    else:
        print(ip, "-- (only FM)", len(num_structures[ip]))
        all_tracker.add(ip_string)
for ip in occus["afm"]:
    ip_string = "".join(sorted([str(ip[0])+str(ip[1]), str(ip[2])+str(ip[3])]))
    if not ip in occus["fm"]:
        print(ip, "-- (only AFM)", len(num_structures[ip]))
        all_tracker.add(ip_string)

print(len(close_to_1_tracker), len(all_tracker), len(close_to_1_tracker)/len(all_tracker))

print("--------------- num structures ------------------------")
print(num_structures)
n = set()
for s in num_structures.values():
    n.update(s)
print("Num structures: ", len(n))

# Display num structures in similar plot
numstructfig = plot_ion_pair_occurrences(occus={"num_structs": {k: len(v) for k, v in num_structures.items()}}, log=False, ion_list=custom_sort_by_electron_configuration)["num_structs"]
numstructfig.update_layout(title=dict(
    text=f"Number of structures, {data_string}, {normalize_string}",
    font=dict(size=10)))
# numstructfig.show()
numstructfig.write_html(os.path.join(plot_dir, f"num_structures_ion_pair_{data_string.replace(' ', '_').replace('°', 'deg')}.html"))

corner connections with TM octahedra at both nodes
{'afm': {('Mn', 3, 'Mn', 3): 208.0, ('Mn', 2, 'Mn', 2): 544.0, ('V', 3, 'V', 3): 200.0, ('Co', 2, 'Co', 2): 184.0, ('Fe', 3, 'Fe', 3): 540.0, ('Ni', 2, 'Ni', 2): 216.0, ('Fe', 2, 'Fe', 2): 240.0, ('Cu', 2, 'Cu', 2): 156.0, ('Cr', 3, 'Cr', 3): 528.0, ('Ru', 4, 'Ru', 4): 64.0, ('Co', 2, 'Os', 6): 32.0, ('Os', 6, 'Co', 2): 32.0, ('Mn', 4, 'Mn', 4): 36.0, ('Os', 5, 'Os', 5): 32.0, ('Fe', 3, 'Cu', 2): 16.0, ('Cu', 2, 'Fe', 3): 16.0, ('Fe', 3, 'Ni', 2): 16.0, ('Ni', 2, 'Fe', 3): 16.0, ('Fe', 2, 'Fe', 3): 128.0, ('Fe', 3, 'Fe', 2): 128.0, ('Mn', 2, 'Fe', 3): 44.0, ('Fe', 3, 'Mn', 2): 44.0, ('Co', 3, 'Co', 3): 76.0, ('Ru', 3, 'Ru', 3): 24.0, ('Fe', 3, 'Re', 5): 24.0, ('Re', 5, 'Fe', 3): 24.0, ('Fe', 3, 'Os', 5): 40.0, ('Os', 5, 'Fe', 3): 40.0, ('Co', 2, 'Ru', 5): 16.0, ('Ru', 5, 'Co', 2): 16.0, ('Ir', 4, 'Ir', 4): 36.0, ('Ni', 2, 'Os', 6): 24.0, ('Os', 6, 'Ni', 2): 24.0, ('Ni', 2, 'Ir', 4): 56.0, ('Ir', 4, 'Ni', 2): 56.0, ('Mn', 2, 'Co', 4): 1

In [6]:
# Repeat only considering oxygen edges

write_mode = "w"
ba_threshold = 150
normalize_bool = False
normalize_string = "absolute occurrences"

data_string = f"oxygen-connected corner connections with TM octahedra at both nodes"
print(data_string)
occus = {"afm": {}, "fm": {}}
num_structures = {}  # track in how many structures ion pair is found with these conditions

missed = set()

for md_id, ang_df in all_stats.items():
    subdf = ang_df.loc[(ang_df["site_is_tm"])  & (ang_df["site_to_is_tm"])]
    subdf = subdf.loc[subdf[f"connectivity"]=="corner"]
    subdf = subdf.loc[(subdf["site_ce"]=="O:6")
            & (subdf["site_to_ce"]=="O:6")]
    subdf = subdf.loc[subdf["ligand_el_set"]=={"O"}]

    if not subdf.empty:
        n_lattice_points = df.at[md_id, "n_lattice_points"]
        for mag_type, condition in zip(["afm", "fm", ],
                                        [(subdf["spin_angle"]>170), (subdf["spin_angle"]<10)]):
            type_df = subdf.loc[condition]
            if not type_df.empty:
                abs_ion_pairs = get_ion_pair_occus(df=type_df, n_lattice_points=n_lattice_points, normalize_bool=normalize_bool)

                for k, v in abs_ion_pairs.items():
                    # Assert no detected ions are missing
                    try:
                        assert (k[0], k[1]) in custom_sort_by_electron_configuration, str((k[0], k[1]))
                        assert (k[2], k[3]) in custom_sort_by_electron_configuration, str((k[2], k[3]))
                    except AssertionError as e:
                        missed.add(str(e))

                    if k in occus[mag_type]:
                        occus[mag_type][k] += v
                    else:
                        occus[mag_type][k] = v

                    if k in num_structures:
                        num_structures[k].add(md_id)
                    else:
                        num_structures[k] = {md_id}



print(occus)

print("missed", missed)

# Plot heatmap of FM/AFM ratio per ion pair (logarithmic, diverging color scale)
ratio_fig = plot_ion_pair_occurrence_ratio(occus=occus, ion_list=custom_sort_by_electron_configuration)
ratio_fig.update_layout(title=dict(
    text=f"Logarithmic plot of FM/AFM ratios, {data_string}, {normalize_string}",
    font=dict(size=10)))
# ratio_fig.show()
ratio_fig.write_html(os.path.join(plot_dir, f"log_plot_afm-fm-ratio_ion_pair_{normalize_string.replace(' ', '_')}_{data_string.replace(' ', '_').replace('°', 'deg')}.html"))

close_to_1_tracker, all_tracker = set(), set()
print("FM to AFM ratio  :")

for ip in occus["fm"]:
    ip_string = "".join(sorted([str(ip[0])+str(ip[1]), str(ip[2])+str(ip[3])]))
    if ip in occus["afm"]:
        print(ip, round(occus["fm"][ip] / occus["afm"][ip], 2), round(log10(occus["fm"][ip] / occus["afm"][ip]), 2), len(num_structures[ip]))
        if abs(log10(occus["fm"][ip] / occus["afm"][ip])) < 0.5:
            close_to_1_tracker.add(ip_string)
        all_tracker.add(ip_string)
    else:
        print(ip, "-- (only FM)", len(num_structures[ip]))
        all_tracker.add(ip_string)
for ip in occus["afm"]:
    ip_string = "".join(sorted([str(ip[0])+str(ip[1]), str(ip[2])+str(ip[3])]))
    if not ip in occus["fm"]:
        print(ip, "-- (only AFM)", len(num_structures[ip]))
        all_tracker.add(ip_string)

print(len(close_to_1_tracker), len(all_tracker), len(close_to_1_tracker)/len(all_tracker))

print("--------------- num structures ------------------------")
print(num_structures)
n = set()
for s in num_structures.values():
    n.update(s)
print("Num structures: ", len(n))

# Display num structures in similar plot
numstructfig = plot_ion_pair_occurrences(occus={"num_structs": {k: len(v) for k, v in num_structures.items()}}, log=False, ion_list=custom_sort_by_electron_configuration)["num_structs"]
numstructfig.update_layout(title=dict(
    text=f"Number of structures, {data_string}, {normalize_string}",
    font=dict(size=10)))
# numstructfig.show()
numstructfig.write_html(os.path.join(plot_dir, f"num_structures_ion_pair_{data_string.replace(' ', '_').replace('°', 'deg')}.html"))

oxygen-connected corner connections with TM octahedra at both nodes
{'afm': {('Mn', 3, 'Mn', 3): 160.0, ('Mn', 2, 'Mn', 2): 260.0, ('V', 3, 'V', 3): 200.0, ('Co', 2, 'Co', 2): 120.0, ('Fe', 3, 'Fe', 3): 468.0, ('Ni', 2, 'Ni', 2): 164.0, ('Fe', 2, 'Fe', 2): 144.0, ('Cu', 2, 'Cu', 2): 136.0, ('Cr', 3, 'Cr', 3): 376.0, ('Ru', 4, 'Ru', 4): 64.0, ('Co', 2, 'Os', 6): 32.0, ('Os', 6, 'Co', 2): 32.0, ('Mn', 4, 'Mn', 4): 36.0, ('Os', 5, 'Os', 5): 32.0, ('Fe', 3, 'Cu', 2): 16.0, ('Cu', 2, 'Fe', 3): 16.0, ('Fe', 3, 'Ni', 2): 16.0, ('Ni', 2, 'Fe', 3): 16.0, ('Fe', 2, 'Fe', 3): 128.0, ('Fe', 3, 'Fe', 2): 128.0, ('Ru', 3, 'Ru', 3): 24.0, ('Fe', 3, 'Re', 5): 24.0, ('Re', 5, 'Fe', 3): 24.0, ('Fe', 3, 'Os', 5): 40.0, ('Os', 5, 'Fe', 3): 40.0, ('Co', 2, 'Ru', 5): 16.0, ('Ru', 5, 'Co', 2): 16.0, ('Ir', 4, 'Ir', 4): 36.0, ('Ni', 2, 'Os', 6): 24.0, ('Os', 6, 'Ni', 2): 24.0, ('Co', 3, 'Co', 3): 16.0, ('Ni', 2, 'Ir', 4): 56.0, ('Ir', 4, 'Ni', 2): 56.0, ('Mn', 2, 'Co', 4): 16.0, ('Co', 4, 'Mn', 2): 16.0, ('Ni